# Bibliotecas

In [ ]:
# Sempre extrair e salvar planinha de IDs

> Dependências: selenium, ipywidgets, IPython, pandas, numpy e openpyxl.


In [ ]:
pip install requests

In [ ]:
import os
import pandas as pd
import numpy as np
import subprocess
import shutil
import requests
import zipfile
import time
import psutil
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions

from regras_pacote_notas import (
    CAUSAS_GERENCIAVEIS,
    FATOR_CLUSTER,
    FATOR_OCORRENCIA_RECENTE,
    JANELA_OCORRENCIAS_RECENTES_DIAS,
    PESO_PRIORIDADE_CONJUNTO,
    BONUS_ALIMENTADOR_345KV,
    converter_numerico_br,
    esperar_download,
    classificar_tipo_nota,
    atribuir_tipo_equipamento,
    calcular_prioridade,
    executar_selecao_por_supervisao,
    META_CSD_PCT,
)


## Configuração e Constantes

In [ ]:
# Regras, constantes e funções compartilhadas são importadas de regras_pacote_notas.py.


## Funções utilitárias

In [ ]:
# converter_numerico_br é compartilhada em regras_pacote_notas.py.


In [ ]:
# Obtém o nome do usuário do sistema
usuario = os.getlogin()
print(usuario)

In [ ]:
# Configuração específica do estado
UF = "ES"
perfil_usuario = os.environ.get("USERPROFILE") or os.path.expanduser("~")
caminho_base = os.path.join(perfil_usuario, "EDP",
    "O365_Planejamento_Manutencao_EDP_Brasil - Documentos",
    "PLANO MANUTENÇÃO", "Pacote de notas", UF, "Base para Geração de Pacotes")
caminho_aux = os.path.join(perfil_usuario, "EDP",
    "O365_Planejamento_Manutencao_EDP_Brasil - Documentos",
    "PLANO MANUTENÇÃO", "Pacote de notas", UF, "Auxiliares")
ARQUIVO_HIERARQUIA = "hierarquia.xlsx"


In [ ]:
#ano atual
ano = datetime.now().year
# Data de hoje
hoje = pd.Timestamp.today().normalize()
print(ano,hoje)

# Baixa base de ocorrencias

In [ ]:
arquivo1 = os.path.join(caminho_base, f"Base_Ocorrencias_{UF}.csv")
arquivo2 = os.path.join(caminho_base, f"Base_Indicadores_Movel_{UF}.csv")

for arquivo in (arquivo1, arquivo2, f"{arquivo1}.crdownload", f"{arquivo2}.crdownload"):
    if os.path.exists(arquivo):
        try:
            os.remove(arquivo)
        except OSError as erro:
            print(f"Não foi possível remover {arquivo}: {erro}")


In [ ]:
# Instala EdgeDriver compatível e baixa a base de ocorrências.
caminho_downloads = caminho_base
for proc in psutil.process_iter(["name"]):
    if proc.info["name"] == "msedgedriver.exe":
        try:
            proc.kill()
        except Exception:
            pass

def acessar():
    global navegador
    resultado = subprocess.run(
        ["reg", "query", "HKEY_CURRENT_USER\\Software\\Microsoft\\Edge\\BLBeacon", "/v", "version"],
        capture_output=True, text=True, shell=True,
    )
    versoes = [linha.split()[-1] for linha in resultado.stdout.splitlines() if "version" in linha]
    if not versoes:
        raise RuntimeError("Não foi possível determinar a versão do Microsoft Edge.")

    download_path = r"C:\Selenium\edgedriver_win64.zip"
    extract_path = r"C:\Selenium"
    os.makedirs(extract_path, exist_ok=True)
    resposta = requests.get(
        f"https://msedgedriver.microsoft.com/{versoes[0]}/edgedriver_win64.zip",
        stream=True, verify=False, timeout=60,
    )
    resposta.raise_for_status()
    with open(download_path, "wb") as arquivo_zip:
        shutil.copyfileobj(resposta.raw, arquivo_zip)
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    os.remove(download_path)

    opcoes = EdgeOptions()
    opcoes.add_argument("--start-maximized")
    opcoes.add_experimental_option("prefs", {
        "download.default_directory": caminho_downloads,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    })
    navegador = webdriver.Edge(service=EdgeService(r"C:\Selenium\msedgedriver.exe"), options=opcoes)
    navegador.get(
        "https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/"
        f"Base_Ocorrencias_{UF}.csv"
    )
    if not esperar_download(arquivo1):
        raise TimeoutError(f"Download não concluído: {arquivo1}")
    return navegador

acessar()
time.sleep(1)


In [ ]:
time.sleep(5)
navegador.get(
    "https://adb-2199566635981619.19.azuredatabricks.net/ajax-api/2.0/fs/files/Volumes/poseidon_uc/group_uc/ddpm/Danilo/"
    f"Base_Indicadores_Movel_{UF}.csv"
)
if not esperar_download(arquivo2):
    raise TimeoutError(f"Download não concluído: {arquivo2}")


In [ ]:
#fecha navegador
# downloads ja confirmados por esperar_download() acima
navegador.quit()


In [ ]:
Indicadores = pd.read_csv(arquivo2, sep=";", encoding="utf-8")
display(Indicadores)


In [ ]:
Ocorrencias = pd.read_csv(arquivo1, sep=";", encoding="utf-8")
display(Ocorrencias)


## Cria dinamica por blocos

In [ ]:
Ocorrencias = converter_numerico_br(Ocorrencias, ['CI', 'CHI'])

Ocorrencias['CI'] = Ocorrencias['CI'].round().astype('Int64')
Ocorrencias['CHI'] = Ocorrencias['CHI'].astype(float)

Ocorrencias

In [ ]:
Ocorrencias = Ocorrencias[Ocorrencias['TIPO_DESLIGAMENTO'] == 'ACIDENTAL']
print(len(Ocorrencias))
Ocorrencias = Ocorrencias[Ocorrencias['CAUSA'].isin(CAUSAS_GERENCIAVEIS)]

print(len(Ocorrencias))

In [ ]:
# Cria coluna de data a partir de ANO, MES, DIA
Ocorrencias['DATA'] = pd.to_datetime(Ocorrencias[['ANO', 'MES', 'DIA']].astype(str).agg('-'.join, axis=1), format='%Y-%m-%d')

# Filtra ocorrências dentro da janela de reincidência
data_limite = datetime.now() - timedelta(days=JANELA_OCORRENCIAS_RECENTES_DIAS)
df_ultimos_90 = Ocorrencias[Ocorrencias['DATA'] >= data_limite]

# Conta distintos ORDEM_ID na janela por EQ_INTERRUPCAO
qtd_90d = df_ultimos_90.groupby('EQ_INTERRUPCAO')['ORDEM_ID'].nunique().reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias_90d'})

# Agrupa ocorrências por equipamento e junta com a contagem recente
grouped_df = Ocorrencias.groupby('EQ_INTERRUPCAO').agg({'ORDEM_ID': 'nunique', 'CI': 'sum', 'CHI': 'sum'}).reset_index().rename(columns={'ORDEM_ID': 'QTD_Ocorrencias'})
grouped_df = grouped_df.merge(qtd_90d, on='EQ_INTERRUPCAO', how='left').fillna({'QTD_Ocorrencias_90d': 0})

grouped_df = converter_numerico_br(grouped_df, ['QTD_Ocorrencias', 'QTD_Ocorrencias_90d', 'CI', 'CHI'])

grouped_df['QTD_Ocorrencias'] = grouped_df['QTD_Ocorrencias'].astype('Int64')
grouped_df['QTD_Ocorrencias_90d'] = grouped_df['QTD_Ocorrencias_90d'].astype('Int64')
grouped_df['CI'] = grouped_df['CI'].round().astype('Int64')
grouped_df['CHI'] = grouped_df['CHI'].astype(float)

# Chave de junção usa o mesmo sufixo '9' de Protecao_Final
grouped_df['EQ_INTERRUPCAO'] = grouped_df['EQ_INTERRUPCAO'].astype(str) + '9'

Dim_Ocorrencias = grouped_df

display(Dim_Ocorrencias)

# Versão de indicadores paleativa

In [ ]:
Dim_Ocorrencias = pd.read_csv(
    fr'{caminho_base}\Ocorrencias_indicadores_es.csv',
    sep=";",   # tenta ; em vez de ,
    encoding="utf-8"  # se der erro de acento, usar "latin1"
)

Dim_Ocorrencias['EQ_INTERRUPCAO'] = Dim_Ocorrencias['EQ_INTERRUPCAO'].astype(str) + '9'

# Colunas inteiras
colunas_int = ['QTD_Ocorrencias', 'QTD_Ocorrencias_90d', 'CI']

# Colunas decimais
colunas_float = ['CHI', 'DEC_EMPRESA', 'FEC_EMPRESA']

Dim_Ocorrencias[colunas_float] = (
    Dim_Ocorrencias[colunas_float]
    .replace(',', '.', regex=True)
    .apply(pd.to_numeric, errors='coerce')
)
# Conversão
Dim_Ocorrencias[colunas_int] = (
    Dim_Ocorrencias[colunas_int]
    .apply(pd.to_numeric, errors='coerce')
    .astype('Int64')  # inteiro que aceita NaN
)

Dim_Ocorrencias[colunas_float] = (
    Dim_Ocorrencias[colunas_float]
    .apply(pd.to_numeric, errors='coerce')
    .astype(float)
)
display(Dim_Ocorrencias)

In [ ]:
Dim_Ocorrencias.columns

# Base de notas

## Le base de Notas

In [ ]:
#Baixar tabelas de ids pendentes 

In [ ]:
Notas = pd.read_excel(os.path.join(caminho_base, "Notas_Pendentes.xlsx"))
display(Notas)


In [ ]:
# Mantém notas pendentes de manutenção ou poda e classifica toda a base processada.
Notas = Notas[Notas["descrição_conjunto"].str.contains("MANUT.|PODA", case=False, na=False)]
Notas = Notas[Notas["Data_encerramento_exec"].isnull()]
Notas = classificar_tipo_nota(Notas)


In [ ]:
# Criando coluna com diferença em dias
Notas['tempo de nota'] = (hoje - Notas['recebido']).dt.days
display(Notas)

In [ ]:
condicoes = [
    Notas['tempo de nota'] <= 365,
    (Notas['tempo de nota'] > 365) & (Notas['tempo de nota'] <= 730),
    Notas['tempo de nota'] > 730
]

categorias = [
    'Até 1 ano',
    'Até 2 anos',
    'Mais de 2 anos'
]

Notas['Classificação Tempo'] = np.select(condicoes, categorias, default='Sem data')
Notas = Notas[Notas['Classificação Tempo'].isin(['Até 1 ano','Até 2 anos'])]

In [ ]:
Notas_bk = Notas.copy()
#Notas = Notas_bk

In [ ]:
len(Notas)

## Remove Medidas que não são manutenção

In [ ]:
Notas = Notas[~Notas['componente_novo'].str.contains('abelha|Aprumar poste|Separador|Placa', case=False, na=False)]

In [ ]:
Notas

# Le Hierarquia

In [ ]:
#tabela é  tabela hierarquia do sql ver posibilidade de pegar do sql 

In [ ]:
hierarquia = pd.read_excel(os.path.join(caminho_aux, ARQUIVO_HIERARQUIA))
display(hierarquia)


# Le capacidade de execução

In [ ]:
capacidade = pd.read_excel(
    os.path.join(caminho_aux, "Capacidade Execução.xlsx"),
    sheet_name="Capacidade_Total",
)
display(capacidade)


In [ ]:
Divisão_Supervisão = pd.read_excel(
    os.path.join(caminho_aux, "Capacidade Execução.xlsx"),
    sheet_name="Divisão por Supervisão",
)
display(Divisão_Supervisão)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

meses = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

titulo = widgets.HTML("<b>Selecione os meses do novo pacote:</b>")
seletor_meses = widgets.SelectMultiple(options=meses, value=(), description='Meses', rows=12)
btn_confirmar = widgets.Button(description="Salvar Meses", button_style="success")
saida = widgets.Output()

meses_selecionados = []

def confirmar_meses(_):
    global meses_selecionados
    meses_selecionados = list(seletor_meses.value)
    with saida:
        saida.clear_output()
        print("Meses selecionados:", meses_selecionados)

btn_confirmar.on_click(confirmar_meses)

display(titulo, seletor_meses, btn_confirmar, saida)


In [ ]:
meses_selecionados = list(seletor_meses.value)

if not meses_selecionados:
    raise ValueError("Selecione pelo menos 1 mês antes de continuar.")

print("Meses selecionados:", meses_selecionados)


In [ ]:
capacidade = capacidade[capacidade['Atributo'].isin(meses_selecionados)]

In [ ]:
capacidade_total = capacidade.groupby('CSD').agg({'Valor': 'sum'}).reset_index()
capacidade_total['META CSD'] = capacidade_total['Valor'] * META_CSD_PCT
display(capacidade_total)

In [ ]:
Divisão_Supervisão_join = Divisão_Supervisão.merge(capacidade_total, left_on='CSD', right_on='CSD', how='left').drop('Valor', axis=1)

Divisão_Supervisão_join['QTD Supervisão'] = np.ceil(
    Divisão_Supervisão_join['META CSD'] *
    Divisão_Supervisão_join['% do CSD']
).astype(int)

Divisão_Supervisão_join['QTD Notas Poda'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Poda']
).astype(int)

Divisão_Supervisão_join['QTD Notas Manut'] = np.ceil(
    Divisão_Supervisão_join['QTD Supervisão'] *
    Divisão_Supervisão_join['% Notas Manut.']
).astype(int)

display(Divisão_Supervisão_join)

# Traz colunas de indicadores para tabela de notas

## Traz colunas de CI,CHI e Ocorrencias

In [ ]:
Notas

In [ ]:
#padroniza coluna de proteção final
Notas['Protecao_Final'] = np.where(
    Notas['dispositivo_protecao'].isna() | 
    (Notas['dispositivo_protecao'].astype(str).str.len() < 3),
    Notas['local_instalacao']+'9',
    Notas['dispositivo_protecao']
)

In [ ]:
Notas_Indicadores = Notas.merge(Dim_Ocorrencias, left_on='Protecao_Final', right_on='EQ_INTERRUPCAO', how='left').drop('EQ_INTERRUPCAO', axis=1)
Notas_Indicadores

In [ ]:
#subistitui valores null por 0
Notas_Indicadores = Notas_Indicadores.fillna({'QTD_Ocorrencias': 0, 'CI': 0, 'CHI': 0, 'QTD_Ocorrencias_90d': 0})

# Convertendo a coluna para inteiro
Notas_Indicadores['QTD_Ocorrencias'] = Notas_Indicadores['QTD_Ocorrencias'].astype('int64')
Notas_Indicadores['CI'] = Notas_Indicadores['CI'].astype('int64')
Notas_Indicadores['QTD_Ocorrencias_90d'] = Notas_Indicadores['QTD_Ocorrencias_90d'].astype('int64')
# Convertendo a coluna para decimal
Notas_Indicadores['CHI'] = Notas_Indicadores['CHI'].astype('float64')

Notas_Indicadores

## Cria coluna de Peso do conjunto

In [ ]:
Prioridade_CJ = pd.read_excel(os.path.join(caminho_base, "Prioridade_Conjuntos_ENG.xlsx"))
display(Prioridade_CJ)


In [ ]:
Notas_Indicadores = atribuir_tipo_equipamento(Notas_Indicadores)
Notas_Indicadores


In [ ]:
# Cria Peso_Conj com base na prioridade cadastrada em Prioridade_CJ
# prioridade 1 -> 2.0 | prioridade 2 -> 1.3 | prioridade 3 -> 1.0 (ver PESO_PRIORIDADE_CONJUNTO)

# Normaliza para número (aceita texto como '1', '2', '3')
prioridade_num = pd.to_numeric(Prioridade_CJ['PRIORIDADE'], errors="coerce")

Indicadores["Peso_Conj"] = prioridade_num.map(PESO_PRIORIDADE_CONJUNTO).fillna(1.0)

Indicadores

In [ ]:
peso_conj = (
    Indicadores[['CONJUNTO_DESC', 'Peso_Conj']]
    .rename(columns={'CONJUNTO_DESC': 'CJ_NOME'})
    .drop_duplicates(subset='CJ_NOME')
)
Notas_Indicadores = (
    Notas_Indicadores
    .drop(columns='Peso_Conj', errors='ignore')
    .merge(peso_conj, on='CJ_NOME', how='left')
)
Notas_Indicadores

# Cria Coluna de Cluster Final

In [ ]:
# Ranking: CI real >= 1000, depois CI substituído; os demais ficam na sequência.
Notas_Indicadores = calcular_prioridade(Notas_Indicadores)
Notas_Indicadores


In [ ]:
#salva planinha no computador 
#Notas_Indicadores.to_excel(fr'{caminho_base}Notas_Rank.xlsx',index=False)

In [ ]:
Notas_Indicadores = (
    Notas_Indicadores
    .drop(columns=['CSD_y'], errors='ignore')
    .rename(columns={'CSD_x': 'CSD'})
)

# Seleção das Notas

In [ ]:
# Poda é selecionada por nota; manutenção preserva o agrupamento por Protecao_Final.
Notas_Indicadores, resumo_etapas_df, faltas_df = executar_selecao_por_supervisao(
    Notas_Indicadores,
    Divisão_Supervisão_join,
)

display(resumo_etapas_df.sort_values(["CSD", "Supervisão", "Tipo Nota Selecionada"]))
if faltas_df.empty:
    print("Sem faltas.")
else:
    display(faltas_df.sort_values(["CSD", "Supervisão", "Tipo Nota"]))


# Salva tabelas finais

In [ ]:
Notas_Indicadores

In [ ]:
COLUNAS_FINAL = ["tipo_equipamento", "Tipo Nota"]

notas_export = Notas_Indicadores.drop(columns=["Tipo Nota Selecionada"], errors="ignore")
notas_export = notas_export[
    [c for c in notas_export.columns if c not in COLUNAS_FINAL]
    + [c for c in COLUNAS_FINAL if c in notas_export.columns]
]

notas_export.to_excel(
    os.path.join(caminho_base, "Notas_Selecionadas.xlsx"),
    index=False,
)
print(len(notas_export.columns), list(notas_export.columns[-3:]))

In [ ]:
with pd.ExcelWriter(
    os.path.join(caminho_base, "Resumo_Selecao_Notas.xlsx"),
    engine="openpyxl",
) as writer:
    resumo_etapas_df.to_excel(writer, sheet_name="resumo_etapas", index=False)
    faltas_df.to_excel(writer, sheet_name="faltas", index=False)
